# Keyframe Embeddings From GCS To Zilliz

This Kaggle notebook reads keyframes from Google Cloud Storage, embeds them with OpenCLIP `ViT-B-32/laion2b_s34b_b79k`, writes per-video artifacts compatible with the existing backend ingest format, and upserts vectors directly to Zilliz Cloud.

The GCS bucket is read as a public bucket. Only Zilliz credentials need to be configured in Kaggle Secrets. The default prefix targets L21 because L30 raw videos exist but processed L30 keyframes are not available yet.

In [ ]:
# Run this cell on Kaggle before imports. Internet must be enabled.
!pip install -q open_clip_torch google-cloud-storage pymilvus

In [ ]:
from __future__ import annotations

import io
import json
import math
import os
import re
import tempfile
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

import open_clip
from google.cloud import storage
from pymilvus import DataType, MilvusClient

try:
    from kaggle_secrets import UserSecretsClient
except Exception:
    UserSecretsClient = None

## Configuration

Set `MAX_VIDEOS=1` and `MAX_IMAGES_PER_VIDEO=5` for the first smoke test. Set both to `None` for the full run.

In [ ]:
@dataclass(frozen=True)
class RunConfig:
    output_root: Path = Path('/kaggle/working/embedding_per_video')
    collection_name: str = 'keyframe_embeddings'
    gcs_keyframe_prefix: str = 'processed/keyframes/dataset=ai_challenge_2025/batch=L21/profile=autoshot_v1'
    model_name: str = 'ViT-B-32'
    pretrained: str = 'laion2b_s34b_b79k'
    batch_size: int = 256
    num_workers: int = 2
    download_batch_size: int = 256
    upsert_batch_size: int = 512
    max_videos: int | None = 1
    max_images_per_video: int | None = 5
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'


CFG = RunConfig()
MODEL_FOLDER = 'vit-' + CFG.model_name.replace('/', '-') + '-' + CFG.pretrained.replace('/', '-')
FEATURE_DIR = CFG.output_root / 'features' / MODEL_FOLDER
MAP_DIR = CFG.output_root / 'features' / 'map-keyframes'
DOWNLOAD_ROOT = Path('/kaggle/working/gcs_keyframes')

FEATURE_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)

print('device:', CFG.device)
print('model_folder:', MODEL_FOLDER)
print('output_root:', CFG.output_root)

In [ ]:
def get_secret(name: str, default: str | None = None) -> str | None:
    value = os.getenv(name)
    if value:
        return value
    if UserSecretsClient is not None:
        try:
            secret = UserSecretsClient().get_secret(name)
            if secret:
                return secret
        except Exception:
            pass
    return default


GCS_BUCKET = get_secret('GCS_BUCKET', 'aic_ai_2026')
KEYFRAME_PREFIX = CFG.gcs_keyframe_prefix.strip().strip('/')
GCS_PUBLIC_URL = (get_secret('GCS_PUBLIC_URL', f'https://storage.googleapis.com/{GCS_BUCKET}') or '').rstrip('/')
MILVUS_URI = get_secret('MILVUS_URI')
MILVUS_TOKEN = get_secret('MILVUS_TOKEN')

if not GCS_BUCKET:
    raise RuntimeError('Missing GCS_BUCKET Kaggle Secret or env var.')
if not MILVUS_URI or not MILVUS_TOKEN:
    raise RuntimeError('Missing MILVUS_URI or MILVUS_TOKEN Kaggle Secret.')

print('gcs_bucket:', GCS_BUCKET)
print('keyframe_prefix:', KEYFRAME_PREFIX)
print('milvus_uri_set:', bool(MILVUS_URI))

In [ ]:
FRAME_RE = re.compile(r'f(\d+)', re.IGNORECASE)
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp'}


def posix_join(*parts: str) -> str:
    cleaned = [str(part).strip('/ ') for part in parts if str(part).strip('/ ')]
    return str(PurePosixPath(*cleaned)) if cleaned else ''


def parse_frame_idx(filename: str) -> int:
    match = FRAME_RE.search(filename)
    if not match:
        raise ValueError(f'Cannot infer frame_idx from filename: {filename}')
    return int(match.group(1))


def parse_video_id_from_parts(parts: list[str]) -> str | None:
    for part in parts:
        if part.startswith('video_id='):
            value = part.split('=', 1)[1].strip()
            return value or None
    if parts:
        return parts[0].strip() or None
    return None


def keyframe_id(video_id: str, frame_idx: int) -> str:
    return f'{video_id}_F{frame_idx:06d}'


def public_url(storage_key: str) -> str:
    return f'{GCS_PUBLIC_URL}/{storage_key}'


def list_top_level_prefixes(bucket: storage.Bucket) -> list[str]:
    iterator = bucket.list_blobs(delimiter='/', max_results=100)
    list(iterator)
    return sorted(iterator.prefixes)


def video_id_from_row(row: pd.Series, fallback: str) -> str:
    video_name = str(row.get('video_name') or '').strip()
    if video_name:
        return Path(video_name).stem
    image_path = str(row.get('image_path') or '').strip()
    if image_path:
        parent = PurePosixPath(image_path.replace('\\', '/')).parent.name
        if parent:
            return parent
    return fallback


storage_client = storage.Client.create_anonymous_client()
bucket = storage_client.bucket(GCS_BUCKET)
print('anonymous public GCS client ready')

## Scan GCS Keyframes

In [ ]:
def load_shot_segments_metadata() -> dict[tuple[str, str], dict[str, Any]]:
    candidates = []
    if KEYFRAME_PREFIX:
        candidates.append(posix_join(KEYFRAME_PREFIX, 'shot_segments.csv'))
    candidates.append('shot_segments.csv')

    for key in candidates:
        blob = bucket.blob(key)
        if not blob.exists(storage_client):
            continue
        text = blob.download_as_text(encoding='utf-8')
        raw = pd.read_csv(io.StringIO(text))
        if 'saved' in raw.columns:
            raw = raw[raw['saved'].astype(str).str.lower().isin(['true', '1', 'yes'])].copy()

        metadata: dict[tuple[str, str], dict[str, Any]] = {}
        for row in raw.to_dict(orient='records'):
            image_path = str(row.get('image_path') or '')
            filename = PurePosixPath(image_path.replace('\\', '/')).name
            if not filename:
                continue
            fallback_video_id = PurePosixPath(image_path.replace('\\', '/')).parent.name
            video_id = video_id_from_row(pd.Series(row), fallback_video_id)
            if not video_id:
                continue
            frame_idx = row.get('frame_idx')
            if pd.isna(frame_idx):
                try:
                    frame_idx = parse_frame_idx(filename)
                except ValueError:
                    continue
            metadata[(video_id, filename)] = {
                'frame_idx': int(frame_idx),
                'pts_time': float(row.get('frame_sec')) if row.get('frame_sec') is not None and not pd.isna(row.get('frame_sec')) else np.nan,
                'fps': float(row.get('fps')) if row.get('fps') is not None and not pd.isna(row.get('fps')) else np.nan,
            }
        print('loaded shot_segments metadata:', key, 'rows:', len(metadata))
        return metadata

    print('shot_segments.csv not found on GCS; falling back to filename frame_idx parsing.')
    return {}


shot_metadata = load_shot_segments_metadata()


def scan_keyframes() -> pd.DataFrame:
    prefix = KEYFRAME_PREFIX + '/' if KEYFRAME_PREFIX else ''
    rows: list[dict[str, Any]] = []
    for blob in bucket.list_blobs(prefix=prefix):
        name = blob.name
        if name.endswith('/') or Path(name).suffix.lower() not in IMAGE_SUFFIXES:
            continue
        rel = name[len(prefix):] if prefix and name.startswith(prefix) else name
        parts = rel.split('/')
        if len(parts) < 2:
            continue
        video_id = parse_video_id_from_parts(parts)
        if not video_id:
            continue
        filename = parts[-1]
        meta = shot_metadata.get((video_id, filename), {})
        try:
            frame_idx = int(meta['frame_idx']) if 'frame_idx' in meta else parse_frame_idx(filename)
        except ValueError as exc:
            print('skip:', exc)
            continue
        rows.append({
            'video_id': video_id,
            'filename': filename,
            'frame_idx': frame_idx,
            'storage_key': name,
            'image_uri': f'gs://{GCS_BUCKET}/{name}',
            'image_url': public_url(name),
            'pts_time': meta.get('pts_time', np.nan),
            'fps': meta.get('fps', np.nan),
        })

    if not rows:
        print('No keyframes found under:', f'gs://{GCS_BUCKET}/{prefix}')
        print('Top-level prefixes:', list_top_level_prefixes(bucket))
        raise RuntimeError('No GCS keyframe images found. Check GCS_BUCKET and the notebook CFG.gcs_keyframe_prefix value.')

    df = pd.DataFrame(rows).sort_values(['video_id', 'frame_idx', 'filename']).reset_index(drop=True)
    if CFG.max_videos is not None:
        selected_videos = sorted(df['video_id'].unique())[:CFG.max_videos]
        df = df[df['video_id'].isin(selected_videos)].copy()
    if CFG.max_images_per_video is not None:
        df = df.groupby('video_id', sort=True).head(CFG.max_images_per_video).reset_index(drop=True)
    return df


keyframes_df = scan_keyframes()
print('videos:', keyframes_df['video_id'].nunique())
print('keyframes:', len(keyframes_df))
display(keyframes_df.head())

## Download Images Locally

In [ ]:
def download_images(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    local_paths: list[str] = []
    for row in tqdm(df.itertuples(index=False), total=len(df), desc='Download keyframes'):
        local_path = DOWNLOAD_ROOT / row.video_id / row.filename
        local_path.parent.mkdir(parents=True, exist_ok=True)
        if not local_path.exists():
            bucket.blob(row.storage_key).download_to_filename(str(local_path))
        local_paths.append(str(local_path))
    df['local_image_path'] = local_paths
    return df


keyframes_df = download_images(keyframes_df)
display(keyframes_df.head())

## Load OpenCLIP

In [ ]:
class ImagePathDataset(Dataset):
    def __init__(self, paths: list[str], preprocess):
        self.paths = list(paths)
        self.preprocess = preprocess

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        image = Image.open(self.paths[idx]).convert('RGB')
        return self.preprocess(image)


model, _, preprocess = open_clip.create_model_and_transforms(
    CFG.model_name,
    pretrained=CFG.pretrained,
    device=CFG.device,
)
model.eval()
print('loaded:', CFG.model_name, CFG.pretrained)

## Connect To Zilliz

In [ ]:
milvus = MilvusClient(uri=MILVUS_URI, token=MILVUS_TOKEN)


def ensure_collection(name: str, dim: int) -> None:
    if milvus.has_collection(collection_name=name):
        return
    schema = MilvusClient.create_schema(auto_id=False, enable_dynamic_field=True)
    schema.add_field(field_name='id', datatype=DataType.VARCHAR, is_primary=True, max_length=128)
    schema.add_field(field_name='vector', datatype=DataType.FLOAT_VECTOR, dim=dim)
    index_params = MilvusClient.prepare_index_params()
    index_params.add_index(field_name='vector', index_type='AUTOINDEX', metric_type='COSINE')
    milvus.create_collection(collection_name=name, schema=schema, index_params=index_params)


ensure_collection(CFG.collection_name, 512)
print('collections:', milvus.list_collections())

## Embed, Save Artifacts, And Upsert

In [ ]:
def encode_paths(paths: list[str]) -> np.ndarray:
    loader = DataLoader(
        ImagePathDataset(paths, preprocess),
        batch_size=CFG.batch_size,
        shuffle=False,
        num_workers=CFG.num_workers,
        pin_memory=(CFG.device == 'cuda'),
    )
    batches: list[np.ndarray] = []
    with torch.no_grad():
        for images in loader:
            images = images.to(CFG.device, non_blocking=True)
            emb = model.encode_image(images, normalize=True)
            batches.append(emb.cpu().numpy().astype('float32'))
    if not batches:
        return np.empty((0, 512), dtype='float32')
    return np.concatenate(batches, axis=0)


def upsert_vectors(video_id: str, group: pd.DataFrame, embeddings: np.ndarray) -> int:
    payload: list[dict[str, Any]] = []
    for row, vector in zip(group.itertuples(index=False), embeddings):
        frame_idx = int(row.frame_idx)
        frame_seconds = -1.0 if pd.isna(row.pts_time) else float(row.pts_time)
        timestamp_ms = -1 if frame_seconds < 0 else int(frame_seconds * 1000)
        kf_id = keyframe_id(video_id, frame_idx)
        payload.append({
            'id': kf_id,
            'vector': vector.astype(float).tolist(),
            'keyframe_id': kf_id,
            'video_id': video_id,
            'frame_idx': frame_idx,
            'frame_seconds': frame_seconds,
            'timestamp_ms': timestamp_ms,
            'image_storage_key': row.storage_key,
            'image_uri': row.image_uri,
            'image_url': row.image_url,
            'model_version': MODEL_FOLDER,
        })
    for start in range(0, len(payload), CFG.upsert_batch_size):
        milvus.upsert(collection_name=CFG.collection_name, data=payload[start:start + CFG.upsert_batch_size])
    return len(payload)


start_all = time.time()
video_infos: list[dict[str, Any]] = []
total_upserted = 0

for video_id, group in tqdm(keyframes_df.groupby('video_id', sort=True), desc='Videos'):
    t0 = time.time()
    group = group.sort_values(['frame_idx', 'filename']).reset_index(drop=True)
    embeddings = encode_paths(group['local_image_path'].tolist())
    if embeddings.shape[0] != len(group):
        raise RuntimeError(f'Embedding row mismatch for {video_id}: {embeddings.shape[0]} != {len(group)}')
    if embeddings.shape[1] != 512:
        raise RuntimeError(f'Embedding dim mismatch for {video_id}: {embeddings.shape[1]} != 512')

    feature_path = FEATURE_DIR / f'{video_id}.npy'
    map_path = MAP_DIR / f'{video_id}.csv'
    np.save(feature_path, embeddings)

    map_df = pd.DataFrame({
        'n': np.arange(1, len(group) + 1, dtype=np.int64),
        'pts_time': group['pts_time'].astype(float),
        'fps': group['fps'].astype(float),
        'frame_idx': group['frame_idx'].astype(np.int64),
    })
    map_df.to_csv(map_path, index=False)

    upserted = upsert_vectors(video_id, group, embeddings)
    total_upserted += upserted
    video_infos.append({
        'video_id': video_id,
        'num_keyframes': len(group),
        'embedding_shape': list(embeddings.shape),
        'feature_path': str(feature_path),
        'map_path': str(map_path),
        'seconds': round(time.time() - t0, 3),
        'zilliz_upserted': upserted,
    })

summary = pd.DataFrame(video_infos)
summary_path = CFG.output_root / 'per_video_summary.csv'
summary.to_csv(summary_path, index=False)

print('done_minutes:', round((time.time() - start_all) / 60, 2))
print('total_upserted:', total_upserted)
display(summary.head())

## Validate Artifacts And Write Model Info

In [ ]:
validation_rows: list[dict[str, Any]] = []
for row in summary.itertuples(index=False):
    vectors = np.load(row.feature_path)
    mapping = pd.read_csv(row.map_path)
    norms = np.linalg.norm(vectors, axis=1) if len(vectors) else np.array([])
    validation_rows.append({
        'video_id': row.video_id,
        'npy_rows': int(vectors.shape[0]),
        'map_rows': int(len(mapping)),
        'dim': int(vectors.shape[1]) if vectors.ndim == 2 else None,
        'min_norm': float(norms.min()) if len(norms) else None,
        'max_norm': float(norms.max()) if len(norms) else None,
    })
    assert vectors.shape[0] == len(mapping), row.video_id
    assert vectors.shape[1] == 512, row.video_id
    if len(norms):
        assert np.allclose(norms, 1.0, atol=1e-3), row.video_id

validation = pd.DataFrame(validation_rows)
display(validation.head())

model_info = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'gcs_bucket': GCS_BUCKET,
    'gcs_keyframe_prefix': KEYFRAME_PREFIX,
    'output_root': str(CFG.output_root),
    'feature_dir': str(FEATURE_DIR),
    'map_dir': str(MAP_DIR),
    'model_name': CFG.model_name,
    'pretrained': CFG.pretrained,
    'model_folder': MODEL_FOLDER,
    'device': CFG.device,
    'batch_size': CFG.batch_size,
    'num_workers': CFG.num_workers,
    'l2_normalized': True,
    'dtype': 'float32',
    'embedding_dim': 512,
    'num_videos': int(keyframes_df['video_id'].nunique()),
    'num_keyframes': int(len(keyframes_df)),
    'csv_columns': ['n', 'pts_time', 'fps', 'frame_idx'],
    'npy_layout': 'one file per video; row index = csv n - 1',
    'zilliz_collection': CFG.collection_name,
    'zilliz_upserted': int(total_upserted),
}

info_path = CFG.output_root / 'model_info.json'
info_path.write_text(json.dumps(model_info, indent=2, ensure_ascii=False), encoding='utf-8')
print(info_path)
model_info

## Zilliz Smoke Check

In [ ]:
print('collections:', milvus.list_collections())
try:
    print('collection_stats:', milvus.get_collection_stats(CFG.collection_name))
except Exception as exc:
    print('stats unavailable:', exc)